# CMSC 173 &middot; Machine Learning &mdash; Week 6 Lab
## Model Selection & Evaluation: Measuring a Classifier Honestly

Training accuracy lies. This lab is about judging a model *honestly*: hold out a **test set**,
read a **confusion matrix**, compute **precision/recall/F1** from scratch, and see how sliding
the decision **threshold** trades one against the other &mdash; ending with the **ROC curve** that
sums it all up.

**How this lab works.** Each part = a short **plain-English explainer**, a **code cell**
you run, a **line-by-line walkthrough** of what it did, and an **Answer here** box. The
code does the maths; we *graph* the results so you can see what is going on.

**NumPy + Matplotlib + scikit-learn.** **Not graded.** About 55 minutes.

---
## Part 0 &middot; Setup + a dengue-screening dataset

We simulate a screening test: from two symptoms we predict whether a patient has dengue. Class
is imbalanced (most patients are negative), which is exactly when accuracy misleads.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
rng = np.random.default_rng(173)

n = 600
y = (rng.random(n) < 0.25).astype(int)                # 25% positive (dengue)
# positives tend to have higher symptom scores, with overlap
x1 = rng.normal(np.where(y==1, 2.0, 0.0), 1.3)
x2 = rng.normal(np.where(y==1, 1.5, 0.0), 1.3)
X = np.column_stack([x1, x2])
print('class balance:', np.bincount(y), '(negatives, positives)')

**Reading the code:** each patient has two symptom scores. Positives are *shifted higher* on both
but overlap the negatives &mdash; so no perfect split exists. Only 25% are positive, so a lazy model
that guesses 'negative' for everyone would already be 75% accurate. Keep that number in mind.

---
## Part 1 &middot; Train here, test *there*

The golden rule: never judge a model on the data it trained on. We split into a training set and
a held-out **test set**, fit on train, and measure on test.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)  # (1) 70/30 split
model = LogisticRegression().fit(Xtr, ytr)            # (2) learn only from training data

train_acc = model.score(Xtr, ytr)                     # (3) accuracy on seen data
test_acc  = model.score(Xte, yte)                     # (4) accuracy on UNSEEN data
print(f'accuracy on training data = {train_acc:.3f}')
print(f'accuracy on test data     = {test_acc:.3f}')
print(f'(guess-all-negative would score {1-yte.mean():.3f})')

**Reading the code, line by line:**
- **(1)** `train_test_split(..., test_size=0.3)` randomly holds out 30% as a test set the model
  never sees during training.
- **(2)** `.fit(Xtr, ytr)` trains only on the training portion.
- **(3)&ndash;(4)** we score both. Test accuracy is the honest number. Notice it isn't far above the
  'guess-all-negative' baseline &mdash; a warning that plain accuracy is a weak judge here.

**Answer here:**

1. Test accuracy is only a little above the guess-everything-negative baseline. Why does high
   accuracy feel *unimpressive* on an imbalanced problem like this?
   &rarr; *your answer*

---
## Part 2 &middot; The confusion matrix

Accuracy hides *which* mistakes happen. The **confusion matrix** counts all four cases: true
positives, true negatives, false positives (false alarm), false negatives (a missed case).

In [ ]:
pred = model.predict(Xte)                             # 0/1 predictions on the test set
TP = int(np.sum((pred==1) & (yte==1)))                # correctly caught
TN = int(np.sum((pred==0) & (yte==0)))                # correctly cleared
FP = int(np.sum((pred==1) & (yte==0)))                # false alarm
FN = int(np.sum((pred==0) & (yte==1)))                # MISSED a real case
print(f'                 predicted +   predicted -')
print(f'actual +   {TP:>10}   {FN:>12}   <- {FN} missed dengue cases')
print(f'actual -   {FP:>10}   {TN:>12}')

**Reading the code:** each line counts one square by combining a prediction condition with a truth
condition. The scary square is **FN** &mdash; real dengue cases the model called negative. In
screening, a missed case (FN) is far worse than a false alarm (FP), and the confusion matrix is
the only view that shows them separately.

**Answer here:**

1. In dengue screening, which is the more dangerous error &mdash; a false positive or a false
   negative? Which count would you most want to push down?
   &rarr; *your answer*

---
## Part 3 &middot; Precision, recall, F1 &mdash; from scratch

Three numbers, straight from the confusion matrix:
- **precision** = of those we *flagged*, how many were real? $TP/(TP+FP)$
- **recall** = of the *real* cases, how many did we catch? $TP/(TP+FN)$
- **F1** = their harmonic mean (one number balancing both).

In [ ]:
precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1        = 2 * precision * recall / (precision + recall)
print(f'from scratch : precision={precision:.3f}  recall={recall:.3f}  f1={f1:.3f}')

# check against sklearn's versions
print(f'sklearn      : precision={precision_score(yte,pred):.3f} '
      f'recall={recall_score(yte,pred):.3f}  f1={f1_score(yte,pred):.3f}')

**Reading the code:** we plug the four counts into the three formulas &mdash; no library needed &mdash; then
confirm they match `sklearn`'s. Low **recall** here means the model *misses* many real cases, even
though its accuracy looked fine. That mismatch is the whole reason we use these metrics.

**Answer here:**

1. Is this model's **recall** high or low? In plain words, what does that mean for the patients it
   is screening?
   &rarr; *your answer*

---
## Part 4 &middot; The threshold knob

A classifier really outputs a **probability**; we turn it into 0/1 by comparing to a threshold
(default 0.5). *Lowering* the threshold flags more patients &mdash; catching more real cases (recall up)
but with more false alarms (precision down). Let's sweep it.

In [ ]:
proba = model.predict_proba(Xte)[:, 1]                # probability of 'positive'
thresholds = np.linspace(0.05, 0.95, 19)
prec_list, rec_list = [], []
for t in thresholds:
    p = (proba >= t).astype(int)
    tp = np.sum((p==1)&(yte==1)); fp = np.sum((p==1)&(yte==0)); fn = np.sum((p==0)&(yte==1))
    prec_list.append(tp/(tp+fp) if tp+fp else 1.0)
    rec_list.append(tp/(tp+fn) if tp+fn else 0.0)

plt.figure(figsize=(7,4))
plt.plot(thresholds, prec_list, 'o-', label='precision')
plt.plot(thresholds, rec_list,  's-', label='recall')
plt.xlabel('decision threshold'); plt.ylabel('score')
plt.title('Lower threshold -> more recall, less precision'); plt.legend()
plt.tight_layout(); plt.show()

**Reading the code:** `predict_proba` gives each patient a probability; for 19 thresholds we recount
the confusion matrix and compute precision and recall. The graph shows them **crossing** &mdash; you
cannot max both. Where you set the threshold is a *values* decision, not a maths one: a screening
tool leans toward high recall (catch cases), even at the cost of more false alarms.

**Answer here:**

1. For dengue screening you'd rather over-flag than miss cases. Would you move the threshold
   *up* or *down* from 0.5? What happens to recall when you do?
   &rarr; *your answer*

---
## Part 5 &middot; The ROC curve (the whole story in one picture)

Instead of picking one threshold, the **ROC curve** plots the tradeoff at *every* threshold:
true-positive rate (recall) against false-positive rate. A model that's all the way to the
top-left is perfect; the diagonal is random guessing. The **area under it (AUC)** is a single
threshold-free score.

In [ ]:
order = np.argsort(-proba)                            # (1) sort patients most-likely first
y_sorted = yte[order]
tpr = np.cumsum(y_sorted) / y_sorted.sum()            # (2) recall as we lower the threshold
fpr = np.cumsum(1 - y_sorted) / (1 - y_sorted).sum()  # (3) false-alarm rate likewise
auc = np.trapz(tpr, fpr)                              # (4) area under the curve

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, lw=2, label=f'our model (AUC = {auc:.2f})')
plt.plot([0,1], [0,1], 'k--', label='random guessing')
plt.xlabel('false positive rate'); plt.ylabel('true positive rate (recall)')
plt.title('ROC curve'); plt.legend(); plt.tight_layout(); plt.show()

**Reading the code, line by line:**
- **(1)** sort patients from most-to-least likely positive &mdash; that's the same as sweeping the
  threshold from high to low.
- **(2)&ndash;(3)** as we admit each next patient, the running fraction of real positives caught is the
  TPR, and of negatives wrongly caught is the FPR.
- **(4)** `np.trapz` integrates the curve for the AUC. Our curve bows above the diagonal (AUC > 0.5)
  &mdash; better than chance &mdash; but the gap from the top-left corner is the room left to improve.

**Answer here:**

1. What AUC would a coin-flip model get, and what would a perfect model get? Where does ours sit?
   &rarr; *your answer*

2. Two models have the same accuracy but different AUC. Which is the better *ranker* of patients,
   and why might that matter more than accuracy?
   &rarr; *your answer*

---
## Where you actually are

Set the pace honestly. Replace each `-` with: **solid** / **rusty** / **never really got it**.

| | You |
|---|---|
| Why train/test split matters | - |
| Reading a confusion matrix | - |
| Precision vs recall (in words) | - |
| The threshold's effect on precision/recall | - |
| Reading an ROC curve / AUC | - |

**Which part took longest, and where did you get stuck?**
&rarr; *your answer*

**In one plain sentence: why can a 90%-accurate model still be a bad screening tool?**
&rarr; *your answer*

---
## Stretch &mdash; optional

Required part is done; nothing below is graded.

### Stretch &middot; Move the threshold on purpose

Re-predict using a threshold of **0.3** instead of 0.5 and print the new recall. Did it rise?

In [ ]:
# your code here: p_new = (proba >= 0.3).astype(int); compute and print recall = TP/(TP+FN)


---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to download.

You need a **submit token**: open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token),
sign in, press the button, then paste it when the cell asks. The cell hides what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "cmsc173", 6

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/cmsc173/lab/6/submit"
    )

nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]
token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 6 submission page](https://portal.latarak.com/course/cmsc173/lab/6/submit) and upload it.

Blank cells are fine and guesses are fine. Don't polish this until it hides what you knew.